# Selector: training and choosing the thresholds (v2)
**Ariel Lax** | Corpus2GeoRAG / CollectCorpus

The gate for new pages: how related is a page to the corpus (the Russia–Ukraine war, military focus)?
```
page ─► paragraphs ─► E5 vectors ─► logistic regression ─► probability per paragraph
     ─► page score = share of paragraphs with probability ≥ paragraph threshold
     ─► accept (≥ accept threshold) / review (in between) / reject (< reject threshold)
```
* **Training:** 1,194 paragraphs from 20 hand-picked pages: 10 clearly related (1); 10 not related (0):
  close in topic but not military (sport, culture, ...) or military but not this war (Nagorno-Karabakh, Mosul).
* **Choosing the thresholds:** the existing corpus (2,853 pages, without the training pages) is used as an exam:
  1,900 pages kept by the category filter (1) and 953 removed (0). The selector never trains on them.
* **v2:** two easy negatives (Ukrainian cuisine, Ukrainian language) were replaced by two military pages from
  other wars. Results are saved with a `_v2` suffix and compared with the first run.

### Before running
Put the new **`paragraphs.csv`** in **`MyDrive/selector`** (replace the old one; `eval_pages.jsonl` stays as is).
Runtime → Change runtime type → **T4 GPU**. Runtime → **Run all**, and approve the Drive connection.
Everything is saved to `MyDrive/selector` as it goes; if the session disconnects, **Run all** again continues.

## Setup

In [ ]:
import hashlib, json, os, re, time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (this will be slow)')

In [ ]:
SMOKE = False   # True = a quick check with a small model on 40 pages, in a local folder
VERSION = 'v2'

if SMOKE:
    WORK = 'data/selector'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/selector'
for f in ['paragraphs.csv', 'eval_pages.jsonl']:
    assert os.path.exists(f'{WORK}/{f}'), f'Put {f} in {WORK} (Google Drive: MyDrive/selector)'
print('working folder:', WORK)

## Splitting a page into paragraphs (`build_training_set.py`)
A paragraph is one line of text inside a section; headings, short lines (< 50 characters) and reference-type
sections (References, See also, ...) are skipped. The same function splits training pages and new pages.

In [ ]:
MIN_CHARS = 50
SKIP_SECTIONS = {"References", "See also", "External links", "Further reading", "Notes", "Bibliography",
                 "Sources", "Citations", "Footnotes", "Explanatory notes", "Works cited"}

In [ ]:
def split_paragraphs(content):
    """[(section path, paragraph), ...] following the rules in the module docstring."""
    out, path, skip = [], [], False
    for line in content.split("\n"):
        line = line.strip()
        heading = re.match(r"^(=+)\s*(.*?)\s*=+$", line)
        if heading:
            level = len(heading.group(1)) - 1          # "== X ==" is level 1
            path = path[:level - 1] + [heading.group(2)]
            if level == 1:
                skip = heading.group(2) in SKIP_SECTIONS
            continue
        if not skip and len(line) >= MIN_CHARS:
            out.append((" > ".join(path) or "(intro)", line))
    return out

## The selector (`selector.py`)

In [ ]:
E5_MODEL = "intfloat/multilingual-e5-large"
PARAGRAPH_THRESHOLD = 0.7
ACCEPT_THRESHOLD = 0.30
REJECT_THRESHOLD = 0.10
MIN_RELATED_PARAGRAPHS = 7

In [ ]:
class Selector:
    def __init__(self, encoder, model_name=E5_MODEL, paragraph_threshold=PARAGRAPH_THRESHOLD,
                 accept_threshold=ACCEPT_THRESHOLD, reject_threshold=REJECT_THRESHOLD,
                 min_related_paragraphs=MIN_RELATED_PARAGRAPHS, clf=None):
        self.encoder = encoder              # a sentence_transformers.SentenceTransformer (E5)
        self.model_name = model_name
        self.paragraph_threshold = paragraph_threshold
        self.accept_threshold = accept_threshold
        self.reject_threshold = reject_threshold
        self.min_related_paragraphs = min_related_paragraphs
        self.clf = clf

    # ---------------------------------------------------------------- vectors
    def embed(self, texts, batch_size=64, show_progress_bar=False):
        """E5 vectors; E5 expects the "query: " prefix for this kind of task."""
        return self.encoder.encode(["query: " + t for t in texts], batch_size=batch_size,
                                   normalize_embeddings=True, show_progress_bar=show_progress_bar)

    # ---------------------------------------------------------------- training
    def fit(self, paragraphs, labels, vectors=None):
        """Logistic regression on the paragraph vectors. class_weight="balanced" evens out the
        different number of paragraphs in the two groups."""
        x = self.embed(paragraphs, show_progress_bar=True) if vectors is None else vectors
        self.clf = LogisticRegression(C=10, max_iter=2000, class_weight="balanced").fit(x, labels)
        return self

    # ---------------------------------------------------------------- scoring
    def paragraph_scores(self, paragraphs, vectors=None):
        """Probability (0..1) that each paragraph is related."""
        if vectors is None:
            if not paragraphs:
                return np.array([])
            vectors = self.embed(paragraphs)
        return self.clf.predict_proba(vectors)[:, 1]

    def page_score(self, paragraph_probs, paragraph_threshold=None):
        """Share of the page's paragraphs that count as related (0..1)."""
        t = self.paragraph_threshold if paragraph_threshold is None else paragraph_threshold
        return float(np.mean(np.asarray(paragraph_probs) >= t)) if len(paragraph_probs) else 0.0

    def decision(self, score, related=0):
        """"accept", "review" or "reject" for a page score (and its number of related paragraphs)."""
        if score >= self.accept_threshold or (self.min_related_paragraphs and related >= self.min_related_paragraphs):
            return "accept"
        return "reject" if score < self.reject_threshold else "review"

    def score_page(self, text):
        """Everything for one page's raw text: {"score", "decision", "paragraphs", "related_paragraphs"}."""
        paragraphs = [p for _, p in split_paragraphs(text)]
        probs = self.paragraph_scores(paragraphs)
        score = self.page_score(probs)
        related = int((probs >= self.paragraph_threshold).sum())
        return {"score": round(score, 3), "decision": self.decision(score, related), "paragraphs": len(paragraphs),
                "related_paragraphs": related}

    # ---------------------------------------------------------------- save / load
    def save(self, folder):
        """The classifier and the settings; the E5 model itself is downloaded by name when loading."""
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        joblib.dump(self.clf, folder / "selector_logreg.joblib")
        (folder / "selector_config.json").write_text(json.dumps({
            "model_name": self.model_name, "paragraph_threshold": self.paragraph_threshold,
            "accept_threshold": self.accept_threshold, "reject_threshold": self.reject_threshold,
            "min_related_paragraphs": self.min_related_paragraphs}, indent=2),
            encoding="utf-8")

    @classmethod
    def load(cls, folder, encoder=None):
        folder = Path(folder)
        config = json.loads((folder / "selector_config.json").read_text(encoding="utf-8"))
        if encoder is None:
            from sentence_transformers import SentenceTransformer
            encoder = SentenceTransformer(config["model_name"])
        return cls(encoder, clf=joblib.load(folder / "selector_logreg.joblib"), **config)

## 1. Training paragraphs → E5 vectors (saved to Drive)

In [ ]:
E5 = 'intfloat/multilingual-e5-small' if SMOKE else E5_MODEL
encoder = SentenceTransformer(E5, device='cuda' if torch.cuda.is_available() else 'cpu')
sel = Selector(encoder, model_name=E5)

train = pd.read_csv(f'{WORK}/paragraphs.csv')
# the file name carries a fingerprint of the paragraphs, so vectors of an older training set are never reused
fingerprint = hashlib.md5('\n'.join(train.text).encode('utf-8')).hexdigest()[:8]
cache = f"{WORK}/train_vectors_{E5.split('/')[-1]}_{fingerprint}.npy"
if os.path.exists(cache):
    train_x = np.load(cache)
else:
    train_x = sel.embed(train.text.tolist(), show_progress_bar=True)
    np.save(cache, train_x)
print(train.label.value_counts().to_dict(), train_x.shape)
print(train.groupby('label').page.unique().apply(list).to_dict())

## 2. Train the logistic regression
Two versions: **all** paragraphs (the one we keep), and **without the 14 paragraphs** that the review flagged
(only to measure how much they matter).

In [ ]:
FLAGGED = [['Battle of Kyiv (2022)', 13], ['Battle of Kyiv (2022)', 24], ['Sinking of the Moskva', 33], ['Sinking of the Moskva', 40], ['Second Battle of Donetsk Airport', 26], ['HESA Shahed 136', 22], ['Andriy Shevchenko', 43], ['FC Dynamo Kyiv', 14], ['FC Dynamo Kyiv', 15], ['FC Dynamo Kyiv', 16], ['FC Dynamo Kyiv', 18], ['FC Dynamo Kyiv', 19], ['Kalush Orchestra', 3], ['Kyiv Metro', 65]]
flag_mask = pd.Series(list(zip(train.page, train.paragraph))).isin([tuple(x) for x in FLAGGED]).to_numpy()
print('flagged paragraphs found:', flag_mask.sum(), '/', len(FLAGGED))
variants = {
    'all': Selector(encoder, model_name=E5).fit(None, train.label.to_numpy(), vectors=train_x),
    'without_14': Selector(encoder, model_name=E5).fit(None, train.label[~flag_mask].to_numpy(), vectors=train_x[~flag_mask]),
}
variants['all'].save(f'{WORK}/model')     # the selector that the collection notebook will load
print('saved', os.listdir(f'{WORK}/model'))

## 3. The exam: split the corpus pages into paragraphs → E5 vectors (in chunks, saved to Drive)

In [ ]:
pages = [json.loads(l) for l in open(f'{WORK}/eval_pages.jsonl', encoding='utf-8')]
if SMOKE:   # 20 pages of each label
    pages = [p for p in pages if p['label'] == 1][:20] + [p for p in pages if p['label'] == 0][:20]
para_page, para_text = [], []
for i, p in enumerate(pages):
    for _, text in split_paragraphs(p['content']):
        para_page.append(i); para_text.append(text)
para_page = np.array(para_page)
print(f'{len(pages)} pages, {len(para_text)} paragraphs')

CHUNK = 5000
chunk_dir = Path(WORK) / f"eval_vectors_{E5.split('/')[-1]}"     # the exam didn't change: reuses v1's vectors
chunk_dir.mkdir(exist_ok=True)
parts = []
for start in range(0, len(para_text), CHUNK):
    path = chunk_dir / f'{start:06d}.npy'
    if not path.exists():
        np.save(path, sel.embed(para_text[start:start + CHUNK]))
        print('  vectors', min(start + CHUNK, len(para_text)), '/', len(para_text))
    parts.append(np.load(path))
eval_x = np.vstack(parts)
labels = np.array([p['label'] for p in pages])

## 4. Choosing the thresholds
For every page: the share of its paragraphs above the paragraph threshold. **AUC** (0.5 = guessing, 1 = perfect)
shows how well that share separates the 1,900 kept pages from the 953 removed ones, for each paragraph threshold.

In [ ]:
def roc_auc(y, s):
    ranks = pd.Series(s).rank().to_numpy(); pos = y == 1
    if pos.all() or (~pos).all():
        return float('nan')
    return float((ranks[pos].sum() - pos.sum() * (pos.sum() + 1) / 2) / (pos.sum() * (~pos).sum()))

PARA_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
shares, auc_rows = {}, []
total = np.bincount(para_page, minlength=len(pages))
for name, s in variants.items():
    probs = s.paragraph_scores(None, vectors=eval_x)
    for t in PARA_THRESHOLDS:
        related = np.bincount(para_page, weights=(probs >= t), minlength=len(pages))
        share = np.divide(related, total, out=np.zeros(len(pages)), where=total > 0)
        shares[(name, t)] = share
        auc_rows.append({'variant': name, 'paragraph_threshold': t, 'auc': round(roc_auc(labels, share), 3)})
auc_table = pd.DataFrame(auc_rows).pivot(index='paragraph_threshold', columns='variant', values='auc')
auc_table

With the best paragraph threshold of the `all` version: what each page threshold would do.
**military dropped** = kept pages (1) below the threshold; **junk let in** = removed pages (0) at or above it.

In [ ]:
best_t = auc_table['all'].fillna(0).idxmax()
share = shares[('all', best_t)]
rows = []
for pt in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7]:
    acc = share >= pt
    tp, fp = int((acc & (labels == 1)).sum()), int((acc & (labels == 0)).sum())
    fn = int((~acc & (labels == 1)).sum())
    rows.append({'page_threshold': pt, 'accepted_pages': int(acc.sum()), 'military_dropped': fn, 'junk_let_in': fp,
                 'recall': round(tp / max(tp + fn, 1), 3), 'precision': round(tp / max(tp + fp, 1), 3)})
threshold_table = pd.DataFrame(rows)
print(f'paragraph threshold = {best_t}')
threshold_table

### The three zones
With the thresholds in `selector.py` (accept ≥ `ACCEPT_THRESHOLD`, reject < `REJECT_THRESHOLD`, review in between),
how the exam pages would be split:

In [ ]:
zone = np.where(share >= ACCEPT_THRESHOLD, 'accept', np.where(share < REJECT_THRESHOLD, 'reject', 'review'))
zones = pd.crosstab(pd.Series(zone, name='zone'), pd.Series(np.where(labels == 1, 'kept (1)', 'removed (0)'), name='category filter'),
                    margins=True, margins_name='total')
print(f'paragraph threshold {best_t}, reject < {REJECT_THRESHOLD}, accept >= {ACCEPT_THRESHOLD}')
zones.reindex(['accept', 'review', 'reject', 'total'])

In [ ]:
scores = pd.DataFrame({'title': [p['title'] for p in pages], 'label': labels,
                       'label_status': [p['label_status'] for p in pages], 'paragraphs': total})
for (name, t), s in shares.items():
    scores[f'{name}@{t}'] = np.round(s, 3)
scores['zone'] = zone
scores.to_csv(f'{WORK}/eval_scores_{VERSION}.csv', index=False)
auc_table.to_csv(f'{WORK}/auc_by_paragraph_threshold_{VERSION}.csv')
threshold_table.to_csv(f'{WORK}/page_threshold_table_{VERSION}.csv', index=False)
zones.to_csv(f'{WORK}/zones_{VERSION}.csv')
print('saved the _' + VERSION + ' results to', WORK)

## 5. Compared with the first run (v1)
Did replacing the two negatives help? AUC, and the scores of pages that are military but not this war
(removed by the category filter; v1 scored them highest).

In [ ]:
OTHER_WARS = ['2023 Bryansk Oblast military aircraft crashes', 'Øresund drone incident', 'Operation Eastern Sentry', '2023 Iran drone attacks', 'Battles of Bambari (2020–2021)', 'Battle of Bangassou', 'Abeïbara massacres', 'Deir ez-Zor offensive (September–November 2017)', 'Battle of Khasham', 'Wagner Group activities in Libya']
v1_path = f'{WORK}/eval_scores.csv'
if os.path.exists(v1_path):
    v1 = pd.read_csv(v1_path).set_index('title')
    col = f'all@{best_t}'
    if col in v1.columns:
        print(f'AUC at paragraph threshold {best_t}:  v1 = {roc_auc(v1.label.values, v1[col].values):.3f}   '
              f'{VERSION} = {roc_auc(labels, share):.3f}')
        cmp = pd.DataFrame({'v1': v1[col], VERSION: scores.set_index('title')[col]})
        display(cmp.loc[[t for t in OTHER_WARS if t in cmp.index]].round(3))
else:
    print('no v1 results (eval_scores.csv) in', WORK)

## 6. Where the gate and the category filter disagree (for review)

In [ ]:
col = f'all@{best_t}'
print('Removed by the category filter (0) but scored highest by the selector:')
display(scores[scores.label == 0].nlargest(15, col)[['title', 'label_status', 'paragraphs', col]])
print('Kept by the category filter (1) but scored lowest by the selector:')
display(scores[scores.label == 1].nsmallest(15, col)[['title', 'label_status', 'paragraphs', col]])

## 7. Using the saved selector on one page

In [ ]:
gate = Selector.load(f'{WORK}/model', encoder=encoder)
example = pages[0]
print(example['title'], gate.score_page(example['content']))